In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## CVM - Fundos Imobiliarios - Ativo Passivo

In [0]:
bronze_path_ativo_passivo = "/Volumes/workspace/case_spark_cvm/bronze/cvm_fii_ativo_passivo/"

df_silver_fii_ativo_passivo = ler_ultima_particao_delta(spark, bronze_path_ativo_passivo)

### 1.1 tratemento silver

#### 1.1.1 Normalizando CNPJ

In [0]:
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.regexp_replace(f.col("CNPJ_FUNDO_CLASSE"), "[^0-9]", "")
)

df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.withColumn(
    "CNPJ_FUNDO_CLASSE",
    f.col("CNPJ_FUNDO_CLASSE").cast("long").cast("string")
)

#### 1.1.2 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa 
colunas_obrigatorias = ['Data_Referencia', 'CNPJ_FUNDO_CLASSE']

# Aplicando a filtro para dropar as colunas
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.dropna(subset=colunas_obrigatorias)

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
# Dropando a Data de Processamento da Bronze 
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.drop("data_processamento")

# Criando a Data de Processamento da silver
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:
df_silver_fii_ativo_passivo = df_silver_fii_ativo_passivo \
    .withColumn('cnpj_fundo_classe', f.col('CNPJ_FUNDO_CLASSE').cast(t.StringType())) \
    .withColumn('data_referencia', f.col('Data_Referencia').cast(t.DateType())) \
    .withColumn('versao', f.col('Versao').cast(t.IntegerType())) \
    .withColumn('total_necessidades_liquidez', f.col('Total_Necessidades_Liquidez').cast(t.DecimalType(22, 2))) \
    .withColumn('disponibilidades', f.col('Disponibilidades').cast(t.DecimalType(22, 2))) \
    .withColumn('titulos_publicos', f.col('Titulos_Publicos').cast(t.DecimalType(22, 2))) \
    .withColumn('titulos_privados', f.col('Titulos_Privados').cast(t.DecimalType(22, 2))) \
    .withColumn('fundos_renda_fixa', f.col('Fundos_Renda_Fixa').cast(t.DecimalType(18, 2))) \
    .withColumn('total_investido', f.col('Total_Investido').cast(t.DecimalType(18, 2))) \
    .withColumn('direitos_bens_imoveis', f.col('Direitos_Bens_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('terrenos', f.col('Terrenos').cast(t.DecimalType(18, 2))) \
    .withColumn('imoveis_renda_acabados', f.col('Imoveis_Renda_Acabados').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_renda_construcao', f.col('Imoveis_Renda_Construcao').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_venda_acabados', f.col('Imoveis_Venda_Acabados').cast(t.DecimalType(22, 2))) \
    .withColumn('imoveis_venda_construcao', f.col('Imoveis_Venda_Construcao').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_direitos_reais', f.col('Outros_Direitos_Reais').cast(t.DecimalType(22, 2))) \
    .withColumn('acoes', f.col('Acoes').cast(t.DecimalType(22, 2))) \
    .withColumn('debentures', f.col('Debentures').cast(t.DecimalType(22, 2))) \
    .withColumn('bonus_subscricao', f.col('Bonus_Subscricao').cast(t.DecimalType(22, 2))) \
    .withColumn('certificados_deposito_valores_mobiliarios', f.col('Certificados_Deposito_Valores_Mobiliarios').cast(t.DecimalType(22, 2))) \
    .withColumn('cedulas_debentures', f.col('Cedulas_Debentures').cast(t.DecimalType(12, 2))) \
    .withColumn('fundo_acoes', f.col('Fundo_Acoes').cast(t.DecimalType(22, 2))) \
    .withColumn('fip', f.col('FIP').cast(t.DecimalType(22, 2))) \
    .withColumn('fii', f.col('FII').cast(t.DecimalType(22, 2))) \
    .withColumn('fdic', f.col('FDIC').cast(t.DecimalType(22, 2))) \
    .withColumn('outras_cotas_fi', f.col('Outras_Cotas_FI').cast(t.DecimalType(22, 2))) \
    .withColumn('notas_promissorias', f.col('Notas_Promissorias').cast(t.DecimalType(22, 2))) \
    .withColumn('acoes_sociedades_atividades_fii', f.col('Acoes_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2))) \
    .withColumn('cotas_sociedades_atividades_fii', f.col('Cotas_Sociedades_Atividades_FII').cast(t.DecimalType(22, 2))) \
    .withColumn('cepac', f.col('CEPAC').cast(t.DecimalType(22, 2))) \
    .withColumn('cri', f.col('CRI').cast(t.DecimalType(22, 2))) \
    .withColumn('cri_cra', f.col('CRI_CRA').cast(t.DecimalType(22, 2))) \
    .withColumn('letras_hipotecarias', f.col('Letras_Hipotecarias').cast(t.DecimalType(22, 2))) \
    .withColumn('lci', f.col('LCI').cast(t.DecimalType(22, 2))) \
    .withColumn('lci_lca', f.col('LCI_LCA').cast(t.DecimalType(22, 2))) \
    .withColumn('lig', f.col('LIG').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_mobliarios', f.col('Outros_Valores_Mobliarios').cast(t.DecimalType(22, 2))) \
    .withColumn('valores_receber', f.col('Valores_Receber').cast(t.DecimalType(22, 2))) \
    .withColumn('contas_receber_aluguel', f.col('Contas_Receber_Aluguel').cast(t.DecimalType(22, 2))) \
    .withColumn('contas_receber_venda_imoveis', f.col('Contas_Receber_Venda_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_receber', f.col('Outros_Valores_Receber').cast(t.DecimalType(22, 2))) \
    .withColumn('rendimentos_distribuir', f.col('Rendimentos_Distribuir').cast(t.DecimalType(22, 2))) \
    .withColumn('taxa_administracao_pagar', f.col('Taxa_Administracao_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('taxa_performance_pagar', f.col('Taxa_Performance_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('obrigacoes_aquisicao_imoveis', f.col('Obrigacoes_Aquisicao_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('adiantamento_venda_imoveis', f.col('Adiantamento_Venda_Imoveis').cast(t.DecimalType(22, 2))) \
    .withColumn('adiantamento_alugueis', f.col('Adiantamento_Alugueis').cast(t.DecimalType(22, 2))) \
    .withColumn('obrigacoes_securitizacao_recebiveis', f.col('Obrigacoes_Securitizacao_Recebiveis').cast(t.DecimalType(22, 2))) \
    .withColumn('instrumentos_financeiros_derivativos', f.col('Instrumentos_Financeiros_Derivativos').cast(t.DecimalType(22, 2))) \
    .withColumn('provisoes_contigencias', f.col('Provisoes_Contigencias').cast(t.DecimalType(22, 2))) \
    .withColumn('outros_valores_pagar', f.col('Outros_Valores_Pagar').cast(t.DecimalType(22, 2))) \
    .withColumn('total_passivo', f.col('Total_Passivo').cast(t.DecimalType(22, 2))) \
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))

### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_bronze_cvm_ativo_passivo.write \
    .mode('overwrite')\
    .partitionBy("data_processamento") \
    .format('delta')\
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.case_spark_cvm.silver_cvm_fii_ativo_passivo")